In [1]:
!pip install -U \
  "transformers==4.51.3" \
  "peft==0.17.0" \
  "bitsandbytes>=0.43.0" \
  sentencepiece \
  tqdm

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import os
import json

import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

model_id = "mistralai/Mistral-7B-Instruct-v0.2"
model_path = "drive/MyDrive/cs-224n-project/mistral_dpo_lora"

eval_files = [
    ("drive/MyDrive/cs-224n-project/test50.jsonl", "drive/MyDrive/cs-224n-project/evaluation_results_dpo.jsonl"),
    ("drive/MyDrive/cs-224n-project/test50_jailbreaks.jsonl", "drive/MyDrive/cs-224n-project/jailbreak_results_dpo.jsonl"),
]


def load_model():
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    base_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        ),
        device_map="auto",
    )

    model = PeftModel.from_pretrained(base_model, model_path)
    model.eval()
    return model, tokenizer

def generate(model, tokenizer, prompt_text):
    prompt = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt_text}],
            tokenize=False,
            add_generation_prompt=True,
        )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=160,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(out[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()


def evaluate(model, tokenizer, test_file, output_file):
    with open(test_file, "r", encoding="utf-8") as f:
        test_data = [json.loads(line) for line in f if line.strip()]

    with open(output_file, "a", encoding="utf-8") as output_f:
        for i, item in enumerate(tqdm(test_data)):
            id = item.get("id", "")

            prompt_text = item.get("prompt") or item.get("unsafe_child")
            result = {"id": id, "prompt": prompt_text, "model_response": generate(model, tokenizer, prompt_text)}
            output_f.write(json.dumps(result, ensure_ascii=False) + "\n")
            output_f.flush()


def main():

    model, tokenizer = load_model()

    for test_file, output_file in eval_files:
        evaluate(model, tokenizer, test_file, output_file)


if __name__ == "__main__":
    main()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]